In [ ]:
import numpy as np
import time
from pathlib import Path
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import networkx as nx

# ============================================================
# Combined Reservoir Comparison Notebook
# Runs both Human Connectome and Gaussian reservoirs with identical parameters
# ============================================================

# ============================================================
# SHARED FUNCTIONS
# ============================================================

def lorenz_euler(n_steps, dt=0.01, sigma=10.0, rho=28.0, beta=8/3, state0=None):
    """Generate Lorenz system data using Euler integration."""
    data = np.empty((n_steps, 3))
    state = np.array([1.0, 1.0, 1.0]) if state0 is None else np.array(state0, dtype=float)

    for i in range(n_steps):
        x, y, z = state
        state = state + dt * np.array([
            sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z,
        ])
        data[i] = state

    return data, state


def extract_data(file_path):
    """Extract graph data from GraphML file."""
    file = Path(file_path)
    G = nx.read_graphml(file)
    return G


def setup_human_reservoir(
    file_path,
    J=1.5,
    lambda_reg=3e-6,
    t_thermalization=30,
    t_training=2000,
    sigma=10.0,
    rho=28.0,
    beta=8 / 3,
    dt=0.01,
    input_scale=1.5,
    seed=42,
):
    """Setup reservoir using human connectome data."""
    G = extract_data(file_path)

    N = G.number_of_nodes()
    node_ids = sorted(G.nodes())
    id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

    # Build connectivity matrix from brain graph
    D = np.zeros((N, N))
    for u, v, attrs in G.edges(data=True):
        i, j = id_to_idx[u], id_to_idx[v]
        w = float(attrs.get("number_of_fibers", 1.0))
        D[i, j] = w
        D[j, i] = w

    rho_D = np.max(np.abs(np.linalg.eigvals(D)))
    if rho_D == 0:
        raise ValueError("connectivity matrix has zero spectral radius")
    W = (J / rho_D) * D
    
    print(f"Human connectome reservoir: N={N}, edges={G.number_of_edges()}, SR={np.max(np.abs(np.linalg.eigvals(W))):.4f}")

    # Time step calculations
    washout_steps = round(t_thermalization / dt)
    train_steps = round(t_training / dt)
    total_steps = washout_steps + train_steps

    # Generate Lorenz training data
    lorenz_full, lorenz_last_state = lorenz_euler(
        total_steps, dt=dt, sigma=sigma, rho=rho, beta=beta
    )

    # Normalize input
    u_mean = lorenz_full.mean(axis=0)
    u_std = lorenz_full.std(axis=0)
    u_std[u_std == 0] = 1.0
    lorenz_full_norm = (lorenz_full - u_mean) / u_std
    u_tr = lorenz_full_norm[washout_steps:]

    rng = np.random.default_rng(seed)
    W_in = input_scale * rng.normal(size=(N, 3))

    # Thermalization
    r = np.zeros(N)
    for t in range(washout_steps):
        r = np.tanh(W @ r + W_in @ lorenz_full_norm[t])

    # Training
    R_train = np.empty((train_steps, N))
    for t in range(train_steps):
        r = np.tanh(W @ r + W_in @ u_tr[t])
        R_train[t] = r

    # Train output weights
    RtR = R_train[:-1].T @ R_train[:-1]
    RtU = R_train[:-1].T @ u_tr[1:]
    W_out = np.linalg.solve(RtR + lambda_reg * np.eye(N), RtU)
    
    return W, W_in, W_out, r, u_tr, u_mean, u_std, lorenz_last_state, N


def setup_gaussian_reservoir(
    N=463,
    J=1.5,
    lambda_reg=3e-6,
    t_thermalization=30,
    t_training=2000,
    sigma=10.0,
    rho=28.0,
    beta=8 / 3,
    dt=0.01,
    input_scale=1.5,
    seed=42,
):
    """Setup reservoir with random Gaussian weights."""
    rng = np.random.default_rng(seed)
    
    # Random Gaussian matrix (dense)
    W_gaussian = rng.normal(size=(N, N)) / np.sqrt(N)
    
    # Scale to spectral radius J
    eigvals = np.linalg.eigvals(W_gaussian)
    rho_W = np.max(np.abs(eigvals))
    
    if rho_W == 0:
        raise ValueError("Gaussian matrix has zero spectral radius")
    
    W = (J / rho_W) * W_gaussian
    actual_sr = np.max(np.abs(np.linalg.eigvals(W)))
    print(f"Gaussian reservoir: N={N}, SR={actual_sr:.4f}")

    # Time step calculations
    washout_steps = round(t_thermalization / dt)
    train_steps = round(t_training / dt)
    total_steps = washout_steps + train_steps

    # Generate Lorenz training data
    lorenz_full, lorenz_last_state = lorenz_euler(
        total_steps, dt=dt, sigma=sigma, rho=rho, beta=beta
    )

    # Normalize input
    u_mean = lorenz_full.mean(axis=0)
    u_std = lorenz_full.std(axis=0)
    u_std[u_std == 0] = 1.0
    lorenz_full_norm = (lorenz_full - u_mean) / u_std
    u_tr = lorenz_full_norm[washout_steps:]

    W_in = input_scale * rng.normal(size=(N, 3))

    # Thermalization
    r = np.zeros(N)
    for t in range(washout_steps):
        r = np.tanh(W @ r + W_in @ lorenz_full_norm[t])

    # Training
    R_train = np.empty((train_steps, N))
    for t in range(train_steps):
        r = np.tanh(W @ r + W_in @ u_tr[t])
        R_train[t] = r

    # Train output weights
    RtR = R_train[:-1].T @ R_train[:-1]
    RtU = R_train[:-1].T @ u_tr[1:]
    W_out = np.linalg.solve(RtR + lambda_reg * np.eye(N), RtU)
    
    return W, W_in, W_out, r, u_tr, u_mean, u_std, lorenz_last_state, N


def run_simulation(W, W_in, W_out, r_last, u_mean, u_std, n_steps=200):
    """Run autonomous prediction after training."""
    N = W.shape[0]
    R = np.empty((n_steps, N))
    Y_norm = np.empty((n_steps, 3))
    Y = np.empty((n_steps, 3))

    r = r_last.copy()

    for t in range(n_steps):
        u_hat_norm = r @ W_out
        u_hat = u_hat_norm * u_std + u_mean
        Y_norm[t] = u_hat_norm
        Y[t] = u_hat
        r = np.tanh(W @ r + W_in @ u_hat_norm)
        R[t] = r

    return R, Y_norm, Y


def run_both_reservoirs(
    file_path,
    N_gaussian,
    J,
    lambda_reg,
    t_thermalization,
    t_training,
    t_prediction,
    sigma,
    rho,
    beta,
    dt,
    input_scale,
    seed,
):
    """Run both human and Gaussian reservoirs and return results."""
    washout_steps = round(t_thermalization / dt)
    train_steps = round(t_training / dt)
    n_steps = round(t_prediction / dt)
    
    print("=" * 60)
    print("RUNNING HUMAN CONNECTOME RESERVOIR")
    print("=" * 60)
    
    t0 = time.perf_counter()
    W_h, W_in_h, W_out_h, r_h, u_tr_h, u_mean_h, u_std_h, lorenz_last_state, N_h = setup_human_reservoir(
        file_path=file_path,
        J=J,
        lambda_reg=lambda_reg,
        t_thermalization=t_thermalization,
        t_training=t_training,
        sigma=sigma,
        rho=rho,
        beta=beta,
        dt=dt,
        input_scale=input_scale,
        seed=seed,
    )
    
    R_pred_h, Y_pred_norm_h, Y_pred_h = run_simulation(
        W=W_h, W_in=W_in_h, W_out=W_out_h,
        r_last=r_h, u_mean=u_mean_h, u_std=u_std_h, n_steps=n_steps
    )
    
    lorenz_states_h, _ = lorenz_euler(n_steps, dt=dt, sigma=sigma, rho=rho, beta=beta, state0=lorenz_last_state)
    Y_true_h = lorenz_states_h
    
    t1 = time.perf_counter()
    print(f"Human reservoir total time: {t1 - t0:.3f} s")
    
    print("\n" + "=" * 60)
    print("RUNNING GAUSSIAN RESERVOIR")
    print("=" * 60)
    
    t0 = time.perf_counter()
    W_g, W_in_g, W_out_g, r_g, u_tr_g, u_mean_g, u_std_g, lorenz_last_state_g, N_g = setup_gaussian_reservoir(
        N=N_gaussian,
        J=J,
        lambda_reg=lambda_reg,
        t_thermalization=t_thermalization,
        t_training=t_training,
        sigma=sigma,
        rho=rho,
        beta=beta,
        dt=dt,
        input_scale=input_scale,
        seed=seed,
    )
    
    R_pred_g, Y_pred_norm_g, Y_pred_g = run_simulation(
        W=W_g, W_in=W_in_g, W_out=W_out_g,
        r_last=r_g, u_mean=u_mean_g, u_std=u_std_g, n_steps=n_steps
    )
    
    lorenz_states_g, _ = lorenz_euler(n_steps, dt=dt, sigma=sigma, rho=rho, beta=beta, state0=lorenz_last_state_g)
    Y_true_g = lorenz_states_g
    
    t1 = time.perf_counter()
    print(f"Gaussian reservoir total time: {t1 - t0:.3f} s")
    
    return {
        'human': {
            'W': W_h, 'W_in': W_in_h, 'W_out': W_out_h,
            'Y_pred': Y_pred_h, 'Y_true': Y_true_h,
            'u_mean': u_mean_h, 'u_std': u_std_h,
            'N': N_h
        },
        'gaussian': {
            'W': W_g, 'W_in': W_in_g, 'W_out': W_out_g,
            'Y_pred': Y_pred_g, 'Y_true': Y_true_g,
            'u_mean': u_mean_g, 'u_std': u_std_g,
            'N': N_g
        }
    }


# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_comparison_components(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=0.01, labels=("x", "y", "z")):
    """Plot side-by-side component comparison."""
    t = np.arange(len(Y_pred_h)) * dt
    fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)

    for i, label in enumerate(labels):
        # Human connectome
        axes[i, 0].plot(t, Y_true_h[:, i], label=f"true {label}", lw=1.5, color="black")
        axes[i, 0].plot(t, Y_pred_h[:, i], "--", label=f"pred {label}", lw=1.2, color="tomato")
        axes[i, 0].set_ylabel(label)
        axes[i, 0].legend(loc="upper right")
        axes[i, 0].set_title("Human Connectome Reservoir")
        
        # Gaussian
        axes[i, 1].plot(t, Y_true_g[:, i], label=f"true {label}", lw=1.5, color="black")
        axes[i, 1].plot(t, Y_pred_g[:, i], "--", label=f"pred {label}", lw=1.2, color="steelblue")
        axes[i, 1].set_ylabel(label)
        axes[i, 1].legend(loc="upper right")
        axes[i, 1].set_title("Gaussian Reservoir")

    axes[-1, 0].set_xlabel("t")
    axes[-1, 1].set_xlabel("t")
    fig.suptitle("Lorenz Components: Human Connectome vs Gaussian Reservoir")
    plt.tight_layout()
    plt.show()


def plot_error_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=0.01):
    """Plot error over time for both reservoirs."""
    t = np.arange(len(Y_pred_h)) * dt
    
    # Compute errors (L2 norm of difference)
    error_h = np.sqrt(((Y_pred_h - Y_true_h) ** 2).sum(axis=1))
    error_g = np.sqrt(((Y_pred_g - Y_true_g) ** 2).sum(axis=1))
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    
    # Top: Error over time
    axes[0].plot(t, error_h, label="Human Connectome", lw=1.2, color="tomato")
    axes[0].plot(t, error_g, label="Gaussian", lw=1.2, color="steelblue")
    axes[0].set_xlabel("t")
    axes[0].set_ylabel("L2 Error")
    axes[0].set_title("Prediction Error Over Time")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Bottom: Cumulative error (running mean)
    cum_error_h = np.cumsum(error_h) / np.arange(1, len(error_h) + 1)
    cum_error_g = np.cumsum(error_g) / np.arange(1, len(error_g) + 1)
    
    axes[1].plot(t, cum_error_h, label="Human Connectome", lw=1.5, color="tomato")
    axes[1].plot(t, cum_error_g, label="Gaussian", lw=1.5, color="steelblue")
    axes[1].set_xlabel("t")
    axes[1].set_ylabel("Mean Cumulative Error")
    axes[1].set_title("Cumulative Mean Error")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "=" * 60)
    print("ERROR SUMMARY")
    print("=" * 60)
    print(f"Human Connectome - Final error: {error_h[-1]:.4f}, Mean error: {error_h.mean():.4f}")
    print(f"Gaussian         - Final error: {error_g[-1]:.4f}, Mean error: {error_g.mean():.4f}")


def plot_3d_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g):
    """Plot 3D attractors side by side."""
    fig = plt.figure(figsize=(14, 5))
    
    # Human connectome
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot(*Y_true_h.T, lw=0.5, color="black", label="true")
    ax1.plot(*Y_pred_h.T, lw=1.2, linestyle="--", color="tomato", label="pred")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.set_zlabel("z")
    ax1.set_title("Human Connectome Reservoir")
    ax1.legend()
    
    # Gaussian
    ax2 = fig.add_subplot(122, projection="3d")
    ax2.plot(*Y_true_g.T, lw=0.5, color="black", label="true")
    ax2.plot(*Y_pred_g.T, lw=1.2, linestyle="--", color="steelblue", label="pred")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y")
    ax2.set_zlabel("z")
    ax2.set_title("Gaussian Reservoir")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CONFIGURATION - Same parameters for both reservoirs
# ============================================================

# File path for human connectome data
file_path = Path("../data/repeated_10_scale_250/123824_repeated10_scale250.graphml")

# Number of nodes for Gaussian reservoir (match human connectome)
N_gaussian = 463

# -------------------------
# physical/system parameters
# -------------------------
dt = 0.01
sigma = 10.0
rho = 50.0
beta = 8 / 3

# -------------------------
# reservoir parameters (IDENTICAL for both)
# -------------------------
J = 0.85           # Spectral radius (0.5-0.9 is good for prediction)
input_scale = 0.5
lambda_reg = 3e-6
seed = 42

# -------------------------
# time horizons (not steps)
# -------------------------
t_thermalization = 30.0
t_training = 10000.0
t_prediction = 10.0

print("Configuration:")
print(f"  N_gaussian = {N_gaussian}")
print(f"  J = {J}")
print(f"  input_scale = {input_scale}")
print(f"  lambda_reg = {lambda_reg}")
print(f"  seed = {seed}")
print(f"  t_thermalization = {t_thermalization}")
print(f"  t_training = {t_training}")
print(f"  t_prediction = {t_prediction}")
print(f"  sigma = {sigma}, rho = {rho}, beta = {beta}, dt = {dt}")

In [ ]:
# ============================================================
# RUN BOTH RESERVOIRS
# ============================================================

results = run_both_reservoirs(
    file_path=file_path,
    N_gaussian=N_gaussian,
    J=J,
    lambda_reg=lambda_reg,
    t_thermalization=t_thermalization,
    t_training=t_training,
    t_prediction=t_prediction,
    sigma=sigma,
    rho=rho,
    beta=beta,
    dt=dt,
    input_scale=input_scale,
    seed=seed,
)

# Extract results
Y_pred_h = results['human']['Y_pred']
Y_true_h = results['human']['Y_true']
Y_pred_g = results['gaussian']['Y_pred']
Y_true_g = results['gaussian']['Y_true']

print("\n" + "=" * 60)
print("RESERVOIR SIZES")
print("=" * 60)
print(f"Human Connectome: {results['human']['N']} nodes")
print(f"Gaussian: {results['gaussian']['N']} nodes")

In [ ]:
# ============================================================
# PLOT 1: Component-wise comparison (side by side)
# ============================================================
plot_comparison_components(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=dt)

In [ ]:
# ============================================================
# PLOT 2: Error over time comparison
# ============================================================
plot_error_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=dt)

In [ ]:
# ============================================================
# PLOT 3: 3D attractor comparison
# ============================================================
plot_3d_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g)